# Análise Exploratória de Dados — Processamento de Petróleo no Brasil
**Autor:** Douglas Chaves Moura

**Objetivo:** Diagnosticar, limpar e extrair insights estratégicos sobre a infraestrutura de refino (*downstream*) no Brasil, com foco em participação de mercado, dependência de insumos (nacional vs. importado) e evolução temporal.

---
## 1. Configurações Iniciais e Importação de Bibliotecas
Nesta etapa, carregamos os pacotes necessários para manipulação matricial, engenharia de features e visualização de dados.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML
from pathlib import Path
import openpyxl
import warnings

# Configurações estéticas e de ambiente
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

def fmt_br(x, casas=1):
    """Formatador numérico padrão internacional adaptado para PT-BR."""
    if pd.isna(x) or x == 0:
        return "NaN"
    return f'{x:,.{casas}f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

## 2. Ingestão e Engenharia de Dados (Data Prep)
O tratamento de dados é fundamental para garantir a integridade da análise. As seguintes ações foram executadas no pipeline:
1. **Padronização:** Conversão de colunas para `snake_case` (sem acentos, minúsculas, separadas por underline).
2. **Enriquecimento Temporal:** Criação de variáveis de data para suportar séries temporais contínuas.
3. **Limpeza de Ruído (Quebra Estrutural):** Remoção dos registros da antiga refinaria 'RPCC', mitigando colisões e distorções com a atual '3R POTIGUAR'.

In [3]:
# Carregamento dos dados
df = pd.read_excel(Path.cwd().parent / "dataset" /'processamento_petroleo_brasil.xlsx')

def clean_column_names(df):
    """Padroniza colunas para snake_case sem acentos e caracteres especiais."""
    df.columns = (
        df.columns
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
        .str.lower()
        .str.replace(r'[^a-z0-9]+', '_', regex=True)
        .str.strip('_')
    )
    return df

# Aplicando limpeza estrutural
df = clean_column_names(df)

# Dicionário de mapeamento para meses
meses_map = {
    'JAN': 'Janeiro', 'FEV': 'Fevereiro', 'MAR': 'Março', 
    'ABR': 'Abril', 'MAI': 'Maio', 'JUN': 'Junho',
    'JUL': 'Julho', 'AGO': 'Agosto', 'SET': 'Setembro', 
    'OUT': 'Outubro', 'NOV': 'Novembro', 'DEZ': 'Dezembro'
}

# Tratamento da dimensão de tempo
df['mes'] = df['mes'].astype(str).str.upper().str.strip()
df['mes_extenso'] = df['mes'].map(meses_map)

# Remoção da refinaria 'RPCC' obsoleta
linhas_antes = len(df)
df = df[df['refinaria'].str.upper() != 'RPCC'].copy()
linhas_depois = len(df)
display(HTML(f"<b style='color:#00A8E8'>✓Limpeza RPCC:</b> Removidas <b>{linhas_antes - linhas_depois}</b> linhas obsoletas."))

# Criação de data sintética para gráficos de série temporal
meses_num = {k: f"{i:02d}" for i, (k, v) in enumerate(meses_map.items(), 1)}
df['mes_num'] = df['mes'].map(meses_num)
df['data'] = pd.to_datetime(df['ano'].astype(str) + '-' + df['mes_num'] + '-01', errors='coerce')

## 3. Diagnóstico dos Dados e Perfil Estatístico

In [4]:
# Visão geral da estrutura (tipos de dados e valores nulos)
df.info()

<class 'pandas.DataFrame'>
Index: 22776 entries, 0 to 23135
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   ano                   22776 non-null  int64         
 1   mes                   22776 non-null  str           
 2   unidade_da_federacao  22776 non-null  str           
 3   refinaria             22776 non-null  str           
 4   materia_prima         22776 non-null  str           
 5   processado            22776 non-null  int64         
 6   mes_extenso           22776 non-null  str           
 7   mes_num               22776 non-null  str           
 8   data                  22776 non-null  datetime64[us]
dtypes: datetime64[us](1), int64(2), str(6)
memory usage: 2.7 MB


In [5]:
# Cardinalidade: Quais são os players e as regiões envolvidas?
print("\n" + "="*50)
print("--- Unidades da Federação ---")
print("="*50)
tabela1 = df["unidade_da_federacao"].unique()
display(tabela1)


print("\n" + "="*50)
print("--- Refinarias Operantes ---")
print("="*50)
tabela2 = df["refinaria"].unique()
display(tabela2)


print("\n" + "="*50)
print("--- Tipos de Matéria-Prima ---")
print("="*50)
tabela3 = df["materia_prima"].unique()
display(tabela3)


--- Unidades da Federação ---


<ArrowStringArray>
[              'BAHIA',           'SÃO PAULO',      'RIO DE JANEIRO',
        'MINAS GERAIS',   'RIO GRANDE DO SUL',              'PARANÁ',
          'PERNAMBUCO',            'AMAZONAS',               'CEARÁ',
 'RIO GRANDE DO NORTE']
Length: 10, dtype: str


--- Refinarias Operantes ---


<ArrowStringArray>
[              'DAX OIL',                  'RPBC',                'UNIVEN',
                  'RLAM',                 'REDUC',                 'REGAP',
                 'REFAP',          'RIOGRANDENSE',                 'REVAP',
                 'REPAR',                 'RECAP',            'MANGUINHOS',
                 'RNEST',                'REPLAN',                 'REMAN',
                'LUBNOR',                 'SSOIL',                'REFMAT',
                  'REAM', '3R POTIGUAR (ex-RPCC)']
Length: 20, dtype: str


--- Tipos de Matéria-Prima ---


<ArrowStringArray>
['OUTRAS CARGAS', 'PETRÓLEO IMPORTADO', 'PETRÓLEO NACIONAL']
Length: 3, dtype: str

### 3.1. Agrupamentos e Hierarquias de Negócio
Investigação da volumetria processada segmentada por Estado e Instalação, e contagem de frequência do mix de cargas.

In [6]:
# Top maiores processamentos agrupados por UF e Refinaria
display(
    df.groupby(["unidade_da_federacao", "refinaria"])["processado"]
    .sum()
    .nlargest(10)
    .reset_index()
    .style.format({"processado": fmt_br})
)

,unidade_da_federacao,refinaria,processado
0,SÃO PAULO,REPLAN,"682.701.285,0"
1,SÃO PAULO,REVAP,"453.123.710,0"
2,RIO DE JANEIRO,REDUC,"422.894.383,0"
3,PARANÁ,REPAR,"372.055.200,0"
4,BAHIA,REFMAT,"363.234.201,0"
5,SÃO PAULO,RPBC,"313.593.499,0"
6,MINAS GERAIS,REGAP,"290.282.760,0"
7,RIO GRANDE DO SUL,REFAP,"277.263.405,0"
8,SÃO PAULO,RECAP,"92.516.925,0"
9,BAHIA,RLAM,"76.663.046,0"


In [7]:
# Frequência de tipos de matéria-prima por instalação
display(
    df.groupby(["unidade_da_federacao", "refinaria"])["materia_prima"]
    .value_counts()
    .nlargest(15)
    .to_frame(name="frequencia")
    .reset_index()
)

,unidade_da_federacao,refinaria,materia_prima,frequencia
0,BAHIA,DAX OIL,PETRÓLEO NACIONAL,436
1,CEARÁ,LUBNOR,OUTRAS CARGAS,436
2,CEARÁ,LUBNOR,PETRÓLEO NACIONAL,436
3,MINAS GERAIS,REGAP,OUTRAS CARGAS,436
4,MINAS GERAIS,REGAP,PETRÓLEO NACIONAL,436
5,MINAS GERAIS,REGAP,PETRÓLEO IMPORTADO,436
6,PARANÁ,REPAR,PETRÓLEO IMPORTADO,436
7,PARANÁ,REPAR,OUTRAS CARGAS,436
8,PARANÁ,REPAR,PETRÓLEO NACIONAL,436
9,PERNAMBUCO,RNEST,OUTRAS CARGAS,436


## 4. Visualizações Estratégicas
A partir daqui, traduzimos os números absolutos em narrativas visuais para extração de *insights*.

### 4.1. Share de Matéria-Prima (Visão Macro)
Entendimento da dependência nacional versus importações de petróleo e uso de cargas alternativas no refino global do país.

In [8]:
df_rosca = df.groupby("materia_prima")["processado"].sum().reset_index()

fig_rosca = px.pie(
    df_rosca, 
    values='processado', 
    names='materia_prima', 
    hole=0.55,
    title='Participação Geral do Processamento por Tipo de Carga',
    template='plotly_dark',
    color='materia_prima',
    color_discrete_sequence=["#FBC28A", "#F38370", "#831E70"]
)

# OTIMIZAÇÃO DE UX: Rótulos posicionados externamente com linhas de chamada (evita sobreposição)
fig_rosca.update_traces(
    textposition='outside',
    textinfo='percent+label',
    hovertemplate=
    "<span style='color:white'><b>Matéria Prima:</b> %{label}<br><b>Volume Processado:</b> %{value:,.2f} m³<extra></extra>"
)

fig_rosca.update_layout(
    separators=",.",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    legend=dict(orientation="h", yanchor="bottom", y=-0.19, xanchor="right", x=1.12)
)

fig_rosca.show()

### 4.2. Série Temporal: Evolução e Volatilidade
Avaliando choques de demanda, paradas de manutenção estruturais e a sazonalidade da entrada de insumos ao longo do tempo.

In [10]:
df_linha = df.groupby(['data', 'materia_prima'], as_index=False)['processado'].sum()

df_linha["processado_br"] = df_linha["processado"].apply(fmt_br)

fig_linha = px.line(
    df_linha, 
    x='data', 
    y='processado',
    color="materia_prima",
    custom_data=["processado_br"],
    category_orders={
        "materia_prima": ["Petróleo Nacional", "Petróleo Importado", "Outras Cagas"]},
    color_discrete_sequence=["#FBC28A", "#F38370", "#831E70"],
    title='Evolução Temporal do Processamento de Petróleo (Barris/m³)',
    template='plotly_dark',
    labels={"materia_prima": "Matéria Prima: "}
)

fig_linha.update_layout(
    xaxis_title="Período",
    yaxis_title="Volume Processado (m³)",
    hovermode="x unified"
)

fig_linha.update_traces(
    hovertemplate=
    "<b>Matéria Prima:</b> %{fullData.name}<br>" +
    "<b>Processado:</b> %{customdata[0]} m³" +
    "<extra></extra>"
)

fig_linha.show()

### 4.3. Mix de Matéria-Prima por Estado
Onde está alocado o petróleo importado? Esta visualização identifica quais estados dependem mais de correntes externas para *blending* ou suprimento de déficit logístico.

In [11]:
df_mix = df.groupby(['unidade_da_federacao', 'materia_prima'])['processado'].sum().reset_index()

df_mix["processado_br"] = df_mix["processado"].apply(fmt_br)

# Ordenando o eixo Y pelos estados com maior volume total
ordem_estados = df_mix.groupby('unidade_da_federacao')['processado'].sum().sort_values(ascending=False).index

fig_bar = px.bar(
    df_mix,
    y='unidade_da_federacao',
    x='processado',
    color='materia_prima',
    custom_data=["processado_br"],
    color_discrete_sequence=["#FBC28A", "#F38370", "#831E70"],
    title="Mix de Dependência de Matéria-Prima por Estado",
    labels={"materia_prima": "Matéria Prima: "},
    barmode='stack',
    category_orders={"unidade_da_federacao": ordem_estados}
)

fig_bar.update_layout(
    template="plotly_dark", 
    yaxis_title="Estado", 
    xaxis_title="Volume Total Processado (m³)"
)

fig_bar.update_traces(
    hovertemplate=
    "<span style='color:white'><b>Matéria Prima:</b> %{fullData.name}</span><br>" +
    "<span style='color:white'><b>Processado:</b> %{customdata[0]} m³</span>" +
    "<extra></extra>"
)

fig_bar.show()

### 4.4. Concentração Geográfica e Market Share (Treemap)
Mapeamento hierárquico do risco de concentração de capacidade. Identificação clara dos polos dominantes (Sudeste) e o peso individual de cada refinaria dentro de sua respectiva UF.

In [12]:
# Calculando a volumetria e removendo zeros para otimizar o algoritmo do Treemap
df_share = (
    df.groupby(['unidade_da_federacao', 'refinaria'])['processado']
      .sum()
      .reset_index()
)

df_share = df_share[df_share['processado'] > 0]

# Coluna formatada para PT-BR
df_share["processado_br"] = (
    df_share["processado"]
    .apply(fmt_br)
)

fig_tree = px.treemap(
    df_share,
    path=[px.Constant("Brasil"), 'unidade_da_federacao', 'refinaria'],
    values='processado',
    custom_data=['processado_br'],
    title="Mapa de Concentração do Refino Nacional (Market Share)",
    color='processado',
    color_continuous_scale='Sunsetdark',
    template='plotly_dark'
)

fig_tree.update_traces(
    root_color="lightgrey",
    texttemplate="%{label}<br>%{value:,.0f}",
    hovertemplate=
    "<b>%{label}</b><br>" +
    "<b>Processado:</b> %{customdata[0]} m³<br>" +
    "<b>Participação:</b> %{percentParent:.2%}" +
    "<extra></extra>"
)

fig_tree.update_layout(
    margin=dict(t=50, l=25, r=25, b=25),
    hoverlabel=dict(
        bgcolor="rgba(0,0,0,0.9)",
        font_color="white",
        font_size=13
    ),
    coloraxis_colorbar=dict(
        title="Volume Processado"
    )
)

fig_tree.show()

In [ ]:
# # Exportando o dataset tratado e filtrado para a pronta aplicação no dash
# nome_arquivo_saida = Path.cwd().parent / "processamento_petroleo_filtered.parquet"
# df.to_parquet(nome_arquivo_saida, index=False)

# print(f"Sucesso! Dataset exportado e pronto para o Dash: {nome_arquivo_saida}")